# Stable Diffusion 2.1 full fine-tuning: 100-step generation and adaptive filtering

This notebook defines generator G02, the canonical full-fine-tuning Stable Diffusion 2.1 experiment evaluated at 100 denoising steps. G02 owns the trained checkpoint family and selects its checkpoint independently by applying the declared mean-validation-FID rule to 100-step samples. G01 reuses the same candidate family but performs a separate 50-step validation selection; equality of the final selected weights is an empirical artifact identity that must be verified, not a design assumption.

The workflow verifies the shared SD2.1 and Diffusers assets, stages the real and traditionally augmented training records, resumes or performs U-Net fine-tuning, selects a checkpoint using the validation split, generates 2,722 RAW images per class, and applies the class-conditional adaptive filter to retain 1,361 images per class. Heavy model and image artifacts are stored under `experiments/diffusers/02_sd21_filtered_100steps/`, while metrics, figures, manifests, and runtime telemetry are stored under `results/2_diffusers/02_sd21_filtered_100steps/`. The filtered downstream dataset is exported to `data/synthetic/02_sd21_filtered_100steps/`.

Checkpoint selection, filtering analysis, and reporting in this notebook use validation data only. Test data remain reserved for final classifier evaluation. Canonical generator selection is performed separately by the validation-only unified benchmark.


## 1. Runtime, reproducibility controls, and experiment configuration

This section locates the repository independently of the notebook working directory,
imports the shared utilities, verifies the pinned Diffusers revision, and reports the
active Python, PyTorch, and CUDA environment. Generation devices are discovered at runtime;
the notebook does not depend on a machine-specific GPU UUID.

Re-execution is explicit. `RUN_TRAINING_PHASE`, `RUN_GENERATION_PHASE`,
`RUN_EVALUATION_PHASE`, and `RUN_FILTER_PHASE` are plain booleans, all `False` by default,
so an ordinary Run All neither retrains nor regenerates; each phase runs only when its own
flag is set. Configuration values centralize prompts, seeds, checkpoint
cadence, sampling parameters, dataset sizes, and output paths so that later cells consume
one consistent experiment definition.

Persistent training and generated-image state belongs under `experiments/`; compact
metrics, plots, manifests, and telemetry belong under `results/`. Existing directories are
created if necessary but are not cleared by this setup section.

In [ ]:
# === Unified notebooks/ bootstrap ===
# Works from the project root and every subdirectory under notebooks/.
import sys as _sys
from pathlib import Path as _Path

def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("MammoDiffusion root not found from " + str(_Path.cwd()))

PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

# === End unified bootstrap ===

# Dependency bootstrap and imports
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime
from contextlib import contextmanager
import gc
import hashlib
import importlib
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import warnings
import zipfile

PROJECT_NAME = "MammoDiffusion"
PROJECT_ROOT_OVERRIDE = None  # Colab example: "/content/drive/MyDrive/MammoDiffusion"
EXPERIMENT_NAME = "diffusers/02_sd21_filtered_100steps"

def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_notebooks = (candidate / "notebooks").is_dir() or (candidate / "notebooks").is_dir()
        if candidate.name == project_name or ((candidate / "data").is_dir() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
            return candidate

    fallback_candidates = [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
        Path.home() / "Progetto" / project_name,
    ]
    for candidate in fallback_candidates:
        if candidate.is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        "MammoDiffusion root not found. Run the notebook from the repository "
        "or set PROJECT_ROOT_OVERRIDE."
    )

PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / ("notebooks" if (PROJECT_ROOT / "notebooks").is_dir() else "notebooks")
UTILITY_DIR = NOTEBOOKS_DIR / "utility" if (NOTEBOOKS_DIR / "utility").is_dir() else NOTEBOOKS_DIR
for _path in (UTILITY_DIR, NOTEBOOKS_DIR):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
EXPERIMENT_DIR = EXPERIMENTS_DIR / EXPERIMENT_NAME
from shared_diffusers_assets import (DIFFUSERS_REVISION, SHARED_DIFFUSERS_REPO_DIR, SHARED_SD21_BASE_DIR, ensure_shared_diffusers_repo, ensure_diffusers_editable_install, shared_diffusers_train_script, verify_diffusers_revision)
DIFFUSERS_REPO_DIR = SHARED_DIFFUSERS_REPO_DIR
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

AUTO_INSTALL_PACKAGES = {
    "gdown": "gdown",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "IPython": "ipython",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "safetensors": "safetensors",
    "huggingface_hub": "huggingface_hub",
    "bitsandbytes": "bitsandbytes",
    "prdc": "prdc",
    "tensorboard": "tensorboard",
}
missing_packages = [
    package_name
    for module_name, package_name in AUTO_INSTALL_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

missing_required = [
    module_name for module_name in ["torch"] if importlib.util.find_spec(module_name) is None
]
if missing_required:
    raise ImportError(
        "Required dependencies not found: "
        + ", ".join(missing_required)
        + ". Install PyTorch in the environment before running the notebook."
    )

DIFFUSERS_REPO_DIR = ensure_shared_diffusers_repo()
verify_diffusers_revision(DIFFUSERS_REPO_DIR)
ensure_diffusers_editable_install(DIFFUSERS_REPO_DIR)
TRAIN_SCRIPT = shared_diffusers_train_script(lora=False)
importlib.invalidate_caches()
# Reload the utility because an active kernel may have cached an
# earlier prepare_sd_manifest signature whose third argument was required.
import parallel_generation_utils as _parallel_generation_utils
_parallel_generation_utils = importlib.reload(_parallel_generation_utils)

for name in [
    "HF_HUB_VERBOSITY",
    "TRANSFORMERS_VERBOSITY",
    "DIFFUSERS_VERBOSITY",
    "ACCELERATE_LOG_LEVEL",
]:
    os.environ[name] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline, UNet2DConditionModel
from generative_evaluator import GenerativeEvaluator
from eco_tracker import measure_sustainability
from PIL import Image
from prdc import compute_prdc
from tensorboard.backend.event_processing import event_accumulator
from tqdm.auto import tqdm

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("PROJECT_ROOT:", PROJECT_ROOT)
# Multi-GPU generation is used only by evaluation and final-dataset creation, never training.
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None
GENERATION_SCHEDULER = "dynamic_reservations"
GENERATION_RESERVATION_SIZE = 4
EVALUATION_GENERATION_SCHEDULER = "auto"
SD_CHECKPOINT_TYPE = "full_unet"

def _sd_parallel_generate_checkpoint_jobs(checkpoints, base_model_dir, eval_dir, negative_prompt, positive_prompt, n_gen, inference_steps, guidance_scale, seed, resolution):
    from parallel_generation_utils import SD_SEED_OFFSETS, missing_named_png_indices, prepare_sd_manifest, run_sd_generation_jobs
    jobs = []
    for step, checkpoint_path in checkpoints:
        requests = []
        for class_name, prompt in (("negative", negative_prompt), ("positive", positive_prompt)):
            class_offset = SD_SEED_OFFSETS[f"evaluation:{class_name}"]
            out_dir = Path(eval_dir) / Path(checkpoint_path).name / class_name
            request = {"name": class_name, "class_name": class_name, "phase": "evaluation", "prompt": prompt, "out_dir": str(out_dir), "count": n_gen, "seed": int(seed), "class_offset": class_offset, "inference_steps": int(inference_steps), "guidance_scale": float(guidance_scale), "resolution": int(resolution), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(Path(base_model_dir).resolve())}
            missing = missing_named_png_indices(out_dir, n_gen)
            if missing:
                prepare_sd_manifest(request, str(checkpoint_path))
                requests.append({**request, "indices": missing})
        if requests:
            jobs.append({"label": f"checkpoint {step}", "checkpoint_path": str(checkpoint_path), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(base_model_dir), "requests": requests})
    if not jobs:
        print("Stable Diffusion evaluation: no missing or corrupt image.")
        return []
    return run_sd_generation_jobs(jobs, GENERATION_GPU_DEVICES if PARALLEL_GENERATION else "off", GENERATION_MAX_WORKERS, Path(EXPERIMENT_DIR) / "logs" / "parallel_generation", Path(PROJECT_ROOT), dry_run=False, generation_scheduler=EVALUATION_GENERATION_SCHEDULER, reservation_size=GENERATION_RESERVATION_SIZE)

def _sd_parallel_generate_final(checkpoint_path, base_model_dir, prompt, out_dir, target_total, inference_steps, guidance_scale, seed, resolution, class_name, reused_prefix, n_reused):
    from parallel_generation_utils import SD_SEED_OFFSETS, final_sd_generation_plan, prepare_sd_manifest, run_sd_final_generation
    n_new = int(target_total) - int(n_reused)
    if n_new < 0:
        raise RuntimeError(f"Reused evaluation images ({n_reused}) exceed the target ({target_total}).")
    request = {"name": class_name, "class_name": class_name, "phase": "final_new", "prompt": prompt, "out_dir": str(out_dir), "indices": [], "count": n_new, "seed": int(seed), "class_offset": SD_SEED_OFFSETS[f"final_new:{class_name}"], "inference_steps": int(inference_steps), "guidance_scale": float(guidance_scale), "resolution": int(resolution), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(Path(base_model_dir).resolve())}
    plan = final_sd_generation_plan(Path(out_dir), target_total, reused_prefix)
    missing = plan["missing_gen_indices"]
    request["indices"] = missing
    if not missing:
        return plan
    prepare_sd_manifest(request, str(checkpoint_path))
    run_sd_final_generation(Path(checkpoint_path), SD_CHECKPOINT_TYPE, Path(base_model_dir), [request], GENERATION_GPU_DEVICES if PARALLEL_GENERATION else "off", GENERATION_MAX_WORKERS, Path(EXPERIMENT_DIR) / "logs" / "parallel_generation", Path(PROJECT_ROOT), dry_run=False, generation_scheduler=GENERATION_SCHEDULER, reservation_size=GENERATION_RESERVATION_SIZE)
    return final_sd_generation_plan(Path(out_dir), target_total, reused_prefix)

In [ ]:
# EXPLICIT_PHASE_FLAGS_V1
# One boolean per phase. Every flag is False, so an ordinary Run All reads the
# artifacts already on disk and reports them: it never retrains and never
# regenerates. To do real work, set the flags for the phases you intend to run
# and execute the notebook top to bottom. Flags are independent -- filtering can
# be redone without regenerating the pool it selects from.
#
#   RUN_TRAINING_PHASE    train the model (hours of GPU)
#   RUN_GENERATION_PHASE  sample a full RAW image pool from the selected checkpoint
#   RUN_EVALUATION_PHASE  score checkpoints and record the selection
#   RUN_FILTER_PHASE      re-run the adaptive filter over the existing RAW pool
#
# Leaving a flag False asserts that the phase's artifact is already complete.
# The cells below check that claim and raise if it does not hold, rather than
# reporting a number they did not verify.
RUN_TRAINING_PHASE = False
RUN_GENERATION_PHASE = False
RUN_EVALUATION_PHASE = False
RUN_FILTER_PHASE = False

In [ ]:
# Datasets shared by the project
DATA_DIR = PROJECT_ROOT / "data"
ARCHIVES_DIR = DATA_DIR / "archives"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
DATA_AUG = DATA_DIR / "real_augmented"

PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
AUGMENTED_DRIVE_ID = "1XRc0SxLEPP-zbMJApn4ruaiH8u_rDc-0"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
AUGMENTED_ZIP_PATH = ARCHIVES_DIR / "real_augmented.zip"

# The experiment and Diffusers repository are defined during initial bootstrap
SD21_MODEL_DRIVE_ID = "10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt"
SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / "pretrained_model"
PRETRAINED_MODEL_DIR = SHARED_SD21_BASE_DIR
PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / "archives" / "stable-diffusion-2-1-base.zip"
FORCE_MODEL_REDOWNLOAD = False

HF_CACHE_DIR = PROJECT_ROOT / ".cache" / "huggingface" / "02_sd21_filtered_100steps"
SD_OUTPUT_DIR = EXPERIMENT_DIR / "model"

# Label-conditioned prompts
POSITIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer positive, malignant finding, "
    "suspicious lesion, medical imaging"
)
NEGATIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer negative, no malignant finding, "
    "normal screening mammogram, medical imaging"
)

# Fine-tuning
RESOLUTION = 512
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-5
MAX_TRAIN_STEPS = 8000
CHECKPOINTING_STEPS = 500
CHECKPOINTS_TOTAL_LIMIT = 32
RESUME_FROM_CHECKPOINT = "latest"
TRAIN_SEED = 42

# Evaluation and generation
N_EVAL_IMAGES_PER_CLASS = 100
N_VALIDATION_IMAGES_PER_CLASS = 73
INFERENCE_STEPS = 100
EVAL_GUIDANCE_SCALE = 7.5
EVAL_SEED = 42
PRDC_NEAREST_K = 5
N_FINAL_IMAGES_PER_CLASS = 2722
FINAL_GENERATE_CLASSES = ["positive", "negative"]
N_SELECTED_PER_CLASS = 1361
RAW_MATCHED_SEED = 42
RAW_MATCHED_COUNT = N_SELECTED_PER_CLASS
RAW_MATCHED_ROOT = EXPERIMENT_DIR / "generated_images" / "raw_matched_1361"
NONBLACK_THRESHOLD = 10
FORCE_RECOMPUTE_VALIDATION_COMPARISON = False

VALIDATION_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "val.csv"
TRAIN_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "train.csv"

RESULTS_DIR = PROJECT_ROOT / "results" / "2_diffusers"
RESULTS_02_DIR = RESULTS_DIR / "02_sd21_filtered_100steps"
METRICS_DIR = RESULTS_02_DIR / "metrics"
PLOTS_DIR = RESULTS_02_DIR / "plots"
ECOTRACKER_DIR = RESULTS_02_DIR / "ecotracker"

EVAL_DIR = EXPERIMENT_DIR / "eval_checkpoints"
EVAL_METRICS_PATH = METRICS_DIR / "checkpoint_validation_metrics.json"
EVAL_SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_validation.jsonl"

FINAL_GEN_DIR = EXPERIMENT_DIR / "generated_images" / "final"
FINAL_NEG_DIR = FINAL_GEN_DIR / "negative"
FINAL_POS_DIR = FINAL_GEN_DIR / "positive"
FINAL_DIRS = {"negative": FINAL_NEG_DIR, "positive": FINAL_POS_DIR}
FINAL_CLASS_LABELS = {"negative": 0, "positive": 1}
RAW_MATCHED_DIRS = {
    class_name: RAW_MATCHED_ROOT / class_name
    for class_name in FINAL_GENERATE_CLASSES
}
RAW_MATCHED_MANIFEST_PATHS = {
    class_name: RAW_MATCHED_ROOT / f"{class_name}_manifest.json"
    for class_name in FINAL_GENERATE_CLASSES
}

SYNTHETIC_DIR = DATA_DIR / "synthetic"
SYNTHETIC_FINE_TUNED_DIR = SYNTHETIC_DIR / "02_sd21_filtered_100steps"
FILTERED_DIRS = {
    class_name: SYNTHETIC_FINE_TUNED_DIR / class_name
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_REPORT_PATHS = {
    class_name: METRICS_DIR / f"filter_report_{class_name}_adaptive_mask.csv"
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_SUMMARY_PATHS = {
    class_name: METRICS_DIR / f"filter_summary_{class_name}_adaptive_mask.json"
    for class_name in FINAL_GENERATE_CLASSES
}

VALIDATION_COMPARISON_CSV = METRICS_DIR / "validation_comparison_100_steps_raw_matched_vs_filtered.csv"
VALIDATION_COMPARISON_JSON = METRICS_DIR / "validation_comparison_100_steps_raw_matched_vs_filtered.json"

SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_finetuning.jsonl"
FINAL_GENERATION_LOG = ECOTRACKER_DIR / "sustainability_generation.jsonl"
GENERATION_INFO_PATH = METRICS_DIR / "generation_info.json"

for directory in [
    ARCHIVES_DIR,
    DATA_AUG,
    EXPERIMENT_DIR,
    PRETRAINED_MODEL_ZIP_PATH.parent,
    HF_CACHE_DIR,
    SD_OUTPUT_DIR,
    EVAL_DIR,
    METRICS_DIR,
    PLOTS_DIR,
    ECOTRACKER_DIR,
    FINAL_NEG_DIR,
    FINAL_POS_DIR,
    RAW_MATCHED_ROOT,
    *RAW_MATCHED_DIRS.values(),
    SYNTHETIC_FINE_TUNED_DIR,
    *FILTERED_DIRS.values(),
]:
    directory.mkdir(parents=True, exist_ok=True)

def label_to_prompt(label):
    prompts = {0: NEGATIVE_PROMPT, 1: POSITIVE_PROMPT}
    try:
        return prompts[int(label)]
    except KeyError as exc:
        raise ValueError(f"Invalid label: {label}") from exc

print("Experiment:", EXPERIMENT_NAME)
print("Experiment directory:", EXPERIMENT_DIR)
print("02 results directory:", RESULTS_02_DIR)
print("Filtered-synthetic directory:", SYNTHETIC_FINE_TUNED_DIR)
print("Inference step:", INFERENCE_STEPS)

## 2. Training-data preparation and integrity checks

The training corpus combines the real training split in `data/processed/` with the
positive-only traditional augmentations indexed by `data/real_augmented/metadata.csv`.
Records are resolved according to their declared source, labels and captions are
normalized, required metadata columns are checked, and representative class/source counts
and images are displayed. Validation and test records are not added to the training corpus.

Diffusers expects a self-contained image-folder dataset. Immediately before training,
`stage_training_dataset` therefore copies the declared 3,061 training samples and a newly
constructed metadata file into a temporary directory. The persistent datasets are never
reorganized or duplicated in place, and the staging directory is removed in a `finally`
block even when the subprocess fails. If the local processed or augmented dataset is
unavailable, the preparation logic can restore the configured archive before applying the
same validation checks.

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ("train", "val", "test")
EXPECTED_LABELS = ("0", "1")

def count_images(directory):
    directory = Path(directory)
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in directory.rglob("*")
    ) if directory.is_dir() else 0

def download_zip(drive_id, destination, force=False):
    destination = Path(destination)
    if destination.exists() and not force and zipfile.is_zipfile(destination):
        print("Archive already present:", destination)
        return
    destination.unlink(missing_ok=True)
    gdown.download(id=drive_id, output=str(destination), quiet=False)
    if not destination.exists() or not zipfile.is_zipfile(destination):
        raise RuntimeError(f"Invalid download: {destination}")

def processed_dataset_ready(directory):
    directory = Path(directory)
    return all(
        count_images(directory / split / label) > 0
        for split in EXPECTED_SPLITS
        for label in EXPECTED_LABELS
    )

def find_directory(root, predicate, description):
    root = Path(root)
    for candidate in [root, *(path for path in root.rglob("*") if path.is_dir())]:
        if predicate(candidate):
            return candidate
    raise FileNotFoundError(f"Directory {description} not found under {root}")

def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR):
        print("Processed dataset already ready.")
        return

    download_zip(PROCESSED_DRIVE_ID, PROCESSED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_processed_extract_") as tmp:
        with zipfile.ZipFile(PROCESSED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, processed_dataset_ready, "processed")
        shutil.copytree(source_dir, DATA_PROCESSED_DIR, dirs_exist_ok=True)

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        raise FileNotFoundError("Processed dataset is incomplete after extraction.")

def augmented_dataset_ready():
    return (DATA_AUG / "metadata.csv").is_file() and count_images(DATA_AUG) > 0

def prepare_augmented_dataset():
    if augmented_dataset_ready():
        print("Augmented dataset already ready.")
        return

    download_zip(AUGMENTED_DRIVE_ID, AUGMENTED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_augmented_extract_") as tmp:
        with zipfile.ZipFile(AUGMENTED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(
            tmp,
            lambda path: (path / "metadata.csv").is_file() and count_images(path) > 0,
            "augmented con metadata.csv",
        )
        DATA_AUG.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_dir / "metadata.csv", DATA_AUG / "metadata.csv")
        for image_path in source_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                destination = DATA_AUG / image_path.name
                if not destination.exists():
                    shutil.copy2(image_path, destination)

    if not augmented_dataset_ready():
        raise FileNotFoundError("Augmented dataset is incomplete after extraction.")

def load_training_metadata(data_aug):
    metadata_path = Path(data_aug) / "metadata.csv"
    metadata = pd.read_csv(metadata_path).copy()
    required = {"file_name", "label"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Missing columns in {metadata_path}: {sorted(missing)}")
    metadata["file_name"] = metadata["file_name"].astype(str).str.replace("\\", "/", regex=False)
    metadata["label"] = metadata["label"].astype(int)
    metadata["text"] = metadata["label"].map(label_to_prompt)
    return metadata

def is_valid_training_image(path):
    path = Path(path)
    return (
        path.is_file()
        and not path.is_symlink()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def resolve_training_image_path(row):
    file_name = Path(str(row["file_name"]))
    label = str(int(row["label"]))
    source = str(row.get("source", "")).strip().lower()
    real_candidate = DATA_PROCESSED_DIR / "train" / label / file_name.name
    augmented_candidate = DATA_AUG / file_name.name

    if source == "real":
        candidates = [real_candidate]
    elif source in {"positive_augmentation", "augmentation", "augmented"}:
        candidates = [augmented_candidate]
    else:
        candidates = [augmented_candidate, real_candidate, PROJECT_ROOT / file_name]

    original_value = row.get("original_processed_path")
    if source == "real" and pd.notna(original_value) and str(original_value).strip():
        original = Path(str(original_value))
        if "data" in original.parts:
            candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
        candidates.append(original)

    for candidate in candidates:
        if is_valid_training_image(candidate):
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"No valid image found for file_name={row['file_name']}, source={source}. "
        f"Checked paths: {checked}"
    )

def stage_training_dataset(metadata_df, staging_dir):
    """Copy every sample into a temporary staging area compatible with HF imagefolder."""
    staging_dir = Path(staging_dir)
    staged = metadata_df.copy()
    staged_names = []

    for index, (_, row) in enumerate(staged.iterrows()):
        source_path = resolve_training_image_path(row)
        destination_name = f"image_{index:06d}{source_path.suffix.lower()}"
        shutil.copy2(source_path, staging_dir / destination_name)
        staged_names.append(destination_name)

    staged["file_name"] = staged_names
    staged.to_csv(staging_dir / "metadata.csv", index=False)

    if count_images(staging_dir) != len(staged):
        raise RuntimeError("The temporary staging area does not contain every expected sample.")
    return staged

In [ ]:
prepare_processed_dataset()
prepare_augmented_dataset()
metadata_df = load_training_metadata(DATA_AUG)

print("Training samples:", len(metadata_df))
print("\nDistribuzione label:")
print(metadata_df["label"].value_counts().sort_index())
if "source" in metadata_df.columns:
    print("\nDistribuzione source:")
    print(metadata_df["source"].value_counts())

sample_df = metadata_df.sample(min(6, len(metadata_df)), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for axis, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_path = resolve_training_image_path(row)
    with Image.open(image_path) as image:
        axis.imshow(image.convert("L"), cmap="gray")
    axis.set_title(f'label={row["label"]} | source={row.get("source", "n/a")}')
    axis.axis("off")
for axis in axes.flat[len(sample_df):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Shared SD2.1 model and pinned Diffusers implementation

This section resolves the project-wide Stable Diffusion 2.1 base at
`notebooks/pretrained_model/stable-diffusion-2-1-base` and the shared Diffusers checkout at
`notebooks/utility/diffusers_repo`. The repository revision is verified against the pinned
commit before the official text-to-image training script is used. The model validator
requires the scheduler, tokenizer, text encoder, VAE, U-Net, configuration, and standard
weight aliases; an incomplete restoration fails before training or generation begins.

The shared base is not copied into the experiment directory. G02 writes only its learned
U-Net checkpoint state and experiment-specific artifacts. The section also initializes the
local generative evaluator and runtime telemetry wrapper used below. These notebook-local
Inception-based diagnostics complement, but do not replace, the later unified benchmark
with its frozen Inception and RAD-DINO identities.

In [ ]:
MODEL_WEIGHT_ALIASES = {
    "text_encoder": ("model.fp16.safetensors", "model.safetensors"),
    "unet": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
    "vae": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
}

def has_diffusers_structure(model_dir):
    model_dir = Path(model_dir)
    required = ("model_index.json", "scheduler", "tokenizer", "text_encoder", "vae", "unet")
    return all((model_dir / item).exists() for item in required)

def local_sd_model_ready(model_dir):
    model_dir = Path(model_dir)
    weights = (
        model_dir / "text_encoder" / "model.safetensors",
        model_dir / "unet" / "diffusion_pytorch_model.safetensors",
        model_dir / "vae" / "diffusion_pytorch_model.safetensors",
    )
    return has_diffusers_structure(model_dir) and all(path.is_file() for path in weights)

def create_model_weight_copies(model_dir):
    """Always create physical copies with the standard filenames expected by Diffusers."""
    model_dir = Path(model_dir)
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.exists():
            continue
        if not source_path.is_file():
            raise FileNotFoundError(f"Source weight not found: {source_path}")
        shutil.copy2(source_path, target_path)

def prepare_pretrained_model():
    if local_sd_model_ready(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print("Stable Diffusion 2.1 model already ready.")
        return PRETRAINED_MODEL_DIR.absolute()

    download_zip(SD21_MODEL_DRIVE_ID, PRETRAINED_MODEL_ZIP_PATH, FORCE_MODEL_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_sd21_extract_") as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, has_diffusers_structure, "Diffusers model")
        if PRETRAINED_MODEL_DIR.is_symlink() or PRETRAINED_MODEL_DIR.is_file():
            PRETRAINED_MODEL_DIR.unlink()
        elif PRETRAINED_MODEL_DIR.exists():
            shutil.rmtree(PRETRAINED_MODEL_DIR)
        shutil.copytree(source_dir, PRETRAINED_MODEL_DIR)
        create_model_weight_copies(PRETRAINED_MODEL_DIR)

    if not local_sd_model_ready(PRETRAINED_MODEL_DIR):
        raise FileNotFoundError("Stable Diffusion 2.1 model is incomplete after extraction.")
    return PRETRAINED_MODEL_DIR.absolute()

LOCAL_MODEL_DIR = prepare_pretrained_model()
print("Local model:", LOCAL_MODEL_DIR)

In [ ]:
# Verify the local Diffusers repository and shared references.
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(f"Training script not found: {TRAIN_SCRIPT}")

print("Diffusers revision:", DIFFUSERS_REVISION)
print("Script training:", TRAIN_SCRIPT)

## 4. Full U-Net fine-tuning

The training cell launches the pinned Diffusers text-to-image script on the temporary
512-pixel image-folder staging set. The configuration uses FP16 arithmetic, a batch size of
2, four gradient-accumulation steps (effective batch size 8), gradient checkpointing,
8-bit Adam, a learning rate of $10^{-5}$, and at most 8,000 optimizer steps. Checkpoints are
written every 500 steps under `experiments/diffusers/02_sd21_filtered_100steps/model/`.

`RESUME_FROM_CHECKPOINT="latest"` resumes the most recent valid checkpoint rather than
restarting. `RUN_TRAINING_PHASE = False` prevents an unintended full retraining when reusable state
is present. Subprocess output is streamed to the notebook, the temporary staging set is always
deleted, and a nonzero return code is raised as an error. Timing and resource telemetry are
appended to `results/2_diffusers/02_sd21_filtered_100steps/ecotracker/`; these records are
operational estimates and are not direct wall-socket energy measurements.

In [ ]:
def build_training_command(train_data_dir):
    command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--mixed_precision=fp16",
        "--num_processes=1",
        str(TRAIN_SCRIPT),
        "--pretrained_model_name_or_path", str(LOCAL_MODEL_DIR),
        "--train_data_dir", str(train_data_dir),
        "--image_column", "image",
        "--caption_column", "text",
        "--resolution", str(RESOLUTION),
        "--center_crop",
        "--train_batch_size", str(TRAIN_BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--gradient_checkpointing",
        "--max_train_steps", str(MAX_TRAIN_STEPS),
        "--learning_rate", str(LEARNING_RATE),
        "--lr_scheduler", "constant",
        "--lr_warmup_steps", "0",
        "--max_grad_norm", "1",
        "--use_8bit_adam",
        "--checkpointing_steps", str(CHECKPOINTING_STEPS),
        "--checkpoints_total_limit", str(CHECKPOINTS_TOTAL_LIMIT),
        "--validation_prompts", POSITIVE_PROMPT, NEGATIVE_PROMPT,
        "--validation_epochs", "2",
        "--seed", str(TRAIN_SEED),
        "--cache_dir", str(HF_CACHE_DIR),
        "--output_dir", str(SD_OUTPUT_DIR),
        "--report_to", "tensorboard",
        "--logging_dir", str(SD_OUTPUT_DIR / "logs"),
        "--dataloader_num_workers", "0",
    ]
    if RESUME_FROM_CHECKPOINT is not None:
        command.extend(["--resume_from_checkpoint", RESUME_FROM_CHECKPOINT])
    return command

def build_training_environment():
    import sysconfig

    environment = os.environ.copy()
    environment["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    conda_prefix = Path(sys.prefix)
    site_packages = Path(sysconfig.get_paths()["purelib"])
    cuda_lib_paths = [
        conda_prefix / "lib",
        conda_prefix / "targets" / "x86_64-linux" / "lib",
        *sorted(site_packages.glob("nvidia/*/lib")),
    ]
    existing = environment.get("LD_LIBRARY_PATH", "")
    environment["LD_LIBRARY_PATH"] = os.pathsep.join(
        [str(path) for path in cuda_lib_paths if path.exists()]
        + ([existing] if existing else [])
    )
    return environment

if RUN_TRAINING_PHASE:
    run_label = f"finetune_sd21_to_{MAX_TRAIN_STEPS}_steps"
    if RESUME_FROM_CHECKPOINT is not None:
        run_label += f"_resume_{RESUME_FROM_CHECKPOINT}"
    temporary_training = TemporaryDirectory(prefix="mammo_sd21_train_copy_")
    try:
        temporary_train_dir = Path(temporary_training.name)
        staged_metadata = stage_training_dataset(metadata_df, temporary_train_dir)
        command = build_training_command(temporary_train_dir)

        print("Temporary staging area:", temporary_train_dir)
        print("Samples copied:", len(staged_metadata))
        print("Fine-tuning command:\n", " ".join(map(str, command)))

        with measure_sustainability(label=run_label, sample_interval=0.5) as eco:
            process = subprocess.Popen(
                command,
                env=build_training_environment(),
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            for line in iter(process.stdout.readline, ""):
                print(line, end="", flush=True)
            process.wait()
    finally:
        temporary_training.cleanup()
        print("Temporary staging area removed.")

    record = eco.metrics.to_dict()
    record.update({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "max_train_steps": MAX_TRAIN_STEPS,
        "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
        "resolution": RESOLUTION,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "source_metadata": str(DATA_AUG / "metadata.csv"),
        "staging_strategy": "temporary_copy",
        "output_dir": str(SD_OUTPUT_DIR),
        "returncode": process.returncode,
        "status": "completed" if process.returncode == 0 else "failed",
    })
    with SUSTAINABILITY_LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("Sustainability metrics:", eco.metrics)
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, command)
else:
    _existing_training_checkpoints = sorted(
        path for path in SD_OUTPUT_DIR.glob("checkpoint-*")
        if path.is_dir() and (path / "unet").is_dir()
    )
    if not _existing_training_checkpoints:
        raise RuntimeError(
            "Training is disabled, but no complete U-Net checkpoint is available."
        )
    print(
        "Training phase skipped; validated", len(_existing_training_checkpoints),
        "existing U-Net checkpoints in", SD_OUTPUT_DIR,
    )

### Training-loss diagnostic

This artifact-only cell reads TensorBoard event files from the checkpoint directory and
reconstructs the denoising-loss trajectory without resuming training. The objective measures
noise-prediction error and is useful for identifying instability, divergence, or an
interrupted run; it is not a direct measure of anatomical fidelity or perceptual image
quality. If no compatible event or scalar tag exists, the cell reports the missing evidence
and exits without modifying model state.

Checkpoint comparisons are consequently based on the validation metrics in the next
section, not on the minimum training loss.

In [ ]:
event_files = sorted((SD_OUTPUT_DIR / "logs").rglob("events.out.tfevents.*"))
loss_rows = []

for event_file in event_files:
    try:
        accumulator = event_accumulator.EventAccumulator(
            str(event_file),
            size_guidance={"scalars": 0},
        )
        accumulator.Reload()
        scalar_tags = accumulator.Tags().get("scalars", [])
        preferred_tags = ["train_loss", "loss"]
        loss_tag = next((tag for tag in preferred_tags if tag in scalar_tags), None)
        if loss_tag is None:
            loss_tag = next((tag for tag in scalar_tags if tag.endswith("/loss")), None)
        if loss_tag is None:
            continue
        loss_rows.extend({
            "step": event.step,
            "loss": event.value,
            "wall_time": event.wall_time,
            "source": event_file.name,
            "tag": loss_tag,
        } for event in accumulator.Scalars(loss_tag))
    except Exception as exc:
        print(f"Unreadable TensorBoard event ({event_file.name}): {exc}")

if not loss_rows:
    print("No TensorBoard loss series is available; the plot was not generated.")
else:
    df_loss = (
        pd.DataFrame(loss_rows)
        .sort_values(["step", "wall_time"])
        .drop_duplicates(subset="step", keep="last")
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_loss["step"], df_loss["loss"], linewidth=1.2)
    ax.set(
        title="Training loss recorded by TensorBoard",
        xlabel="Training step",
        ylabel="Loss",
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "train_loss.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Loss curve saved under:", PLOTS_DIR / "train_loss.png")

## 5. Validation-only checkpoint evaluation and selection

Every structurally valid `checkpoint-<step>` U-Net is evaluated against the real validation
split. For each checkpoint, the pipeline generates 100 negative and 100 positive images at
100 denoising steps, then computes class-specific FID, Inception Score, and PRDC statistics
together with their reported averages. Previously generated images and a complete compatible
metrics cache are reused; missing files are generated deterministically, and pipeline memory
is released in `finally` blocks.

The frozen rule selects the checkpoint with the lowest mean validation FID. Test data do not
participate in this decision. Each class contains only 73 real validation references, so FID
and especially neighborhood-based PRDC estimates have appreciable finite-sample uncertainty.
They support comparisons under the same reference and extraction procedure, but their
absolute values should not be treated as population-level estimates. The canonical RQ1
ranking is produced later by the unified validation-only benchmark, not by these preliminary
notebook-local metrics.

Generated checkpoint samples are retained under `experiments/.../eval_checkpoints/`, and
the cache is written to `results/2_diffusers/02_sd21_filtered_100steps/metrics/`.

In [ ]:
# Checkpoint-selection and final-generation configuration
for metadata_path in [VALIDATION_METADATA_PATH, TRAIN_METADATA_PATH]:
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Split metadata not found: {metadata_path}")

print("Source checkpoint       :", SD_OUTPUT_DIR)
print("Validation metadata       :", VALIDATION_METADATA_PATH)
print("Generated images/checkpoint/class:", N_EVAL_IMAGES_PER_CLASS)
print("Final images/class    :", N_FINAL_IMAGES_PER_CLASS)

In [ ]:
# Temporary real validation references, always copied
def resolve_split_image_path(raw_path, split_name, label):
    original = Path(str(raw_path)).expanduser()
    candidates = [original] if original.is_absolute() else [PROJECT_ROOT / original]
    if "data" in original.parts:
        candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
    candidates.append(DATA_PROCESSED_DIR / split_name / str(int(label)) / original.name)

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate.resolve()
    checked = "\n".join(f"  - {candidate}" for candidate in dict.fromkeys(candidates))
    raise FileNotFoundError(f"Image not found for {raw_path}. Checked paths:\n{checked}")

def get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed=42):
    metadata_path = Path(metadata_path)
    metadata = pd.read_csv(metadata_path)
    required = {"label", "processed_path"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Missing columns in {metadata_path}: {sorted(missing)}")

    subset = metadata[metadata["label"].astype(int) == int(label)]
    if subset.empty:
        raise ValueError(f"No image with label={label} in {metadata_path}")
    n_take = len(subset) if n_images is None else min(int(n_images), len(subset))
    if n_images is not None and n_take < n_images:
        print(f"Requested {n_images} images label={label}; {n_take} available.")

    subset = subset.sample(n=n_take, random_state=seed)
    return [
        resolve_split_image_path(raw_path, metadata_path.stem.lower(), label)
        for raw_path in subset["processed_path"]
    ]

@contextmanager
def temporary_real_reference_dir_from_split_metadata(metadata_path, label, n_images, seed=42):
    image_paths = get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed)
    with TemporaryDirectory(prefix=f"real_ref_label{label}_") as tmp:
        tmp_dir = Path(tmp)
        for index, source_path in enumerate(image_paths):
            destination = tmp_dir / f"real_{label}_{index:04d}{source_path.suffix.lower()}"
            shutil.copy2(source_path, destination)
        yield tmp_dir

@contextmanager
def temporary_real_reference_dirs_from_split_metadata(metadata_path, n_per_class, seed=42):
    with temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=0, n_images=n_per_class, seed=seed
    ) as real_neg_dir, temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=1, n_images=n_per_class, seed=seed + 1
    ) as real_pos_dir:
        yield real_neg_dir, real_pos_dir

print("Checkpoint selection: validation.")

In [ ]:
# Checkpoint, generation, and metric utilities
def discover_checkpoints(output_dir):
    checkpoints = []
    for path in Path(output_dir).glob("checkpoint-*"):
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if match and path.is_dir() and (path / "unet").is_dir():
            checkpoints.append((int(match.group(1)), path))
    return sorted(checkpoints)

def load_pipeline_from_checkpoint(checkpoint_path, base_model_dir, device="cuda"):
    unet = UNet2DConditionModel.from_pretrained(
        str(Path(checkpoint_path) / "unet"),
        torch_dtype=torch.float16,
    )
    pipeline = StableDiffusionPipeline.from_pretrained(
        str(base_model_dir),
        unet=unet,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipeline.set_progress_bar_config(disable=True)
    return pipeline

def count_pngs(directory):
    return sum(path.is_file() and not path.name.startswith(".tmp_") for path in Path(directory).glob("*.png")) if Path(directory).exists() else 0

def generate_images_to_dir(
    pipeline, prompt, out_dir, n, inference_steps, guidance_scale, seed, resolution=512
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n_existing = count_pngs(out_dir)
    if n_existing >= n:
        print(f"    {n_existing} images already present; skipping")
        return

    print(f"    {n_existing} already present; generating {n - n_existing} missing")
    generator = torch.Generator("cuda").manual_seed(seed + n_existing)
    for index in tqdm(range(n_existing, n), desc=f"  Generation {out_dir.name}", unit="img"):
        image = pipeline(
            prompt,
            num_inference_steps=inference_steps,
            guidance_scale=guidance_scale,
            height=resolution,
            width=resolution,
            generator=generator,
        ).images[0]
        image.save(out_dir / f"gen_{index:04d}.png")

def compute_prdc_metrics(real_features, fake_features, nearest_k=PRDC_NEAREST_K):
    """Compute PRDC by reusing the Inception features already extracted for FID."""
    metrics = compute_prdc(
        real_features=real_features,
        fake_features=fake_features,
        nearest_k=nearest_k,
    )
    return {
        name: round(float(metrics[name]), 4)
        for name in ("precision", "recall", "density", "coverage")
    }

def eval_one_checkpoint(
    step,
    ckpt_path,
    base_model_dir,
    real_neg_dir,
    real_pos_dir,
    eval_dir,
    n_gen,
    inference_steps,
    guidance_scale,
    seed,
    resolution=512,
):
    class_config = {
        "negative": (NEGATIVE_PROMPT, Path(real_neg_dir), seed),
        "positive": (POSITIVE_PROMPT, Path(real_pos_dir), seed + 1),
    }
    generated_dirs = {
        name: Path(eval_dir) / f"checkpoint-{step}" / name
        for name in class_config
    }

    print(f"\nCheckpoint {step} | {Path(ckpt_path).name}")
    needs_generation = any(count_pngs(path) < n_gen for path in generated_dirs.values())
    pipeline = load_pipeline_from_checkpoint(ckpt_path, base_model_dir) if needs_generation else None
    try:
        for name, (prompt, _, class_seed) in class_config.items():
            if count_pngs(generated_dirs[name]) < n_gen:
                print(f"  Generating {name}...")
                generate_images_to_dir(
                    pipeline,
                    prompt,
                    generated_dirs[name],
                    n_gen,
                    inference_steps,
                    guidance_scale,
                    class_seed,
                    resolution,
                )
    finally:
        if pipeline is not None:
            del pipeline
            gc.collect()
            torch.cuda.empty_cache()

    metrics = {}
    for name, (_, real_dir, _) in class_config.items():
        print(f"  FID + IS + PRDC: {name}")
        evaluator = GenerativeEvaluator(
            real_dir=real_dir,
            generated_dir=generated_dirs[name],
            batch_size=8,
            num_workers=0,
        )
        fid_is_metrics, real_features, fake_features = evaluator.compute_with_features()
        metrics[name] = {
            **fid_is_metrics,
            **compute_prdc_metrics(real_features, fake_features),
        }

    averages = {
        metric: round(sum(metrics[name][metric] for name in metrics) / len(metrics), 4)
        for metric in ("FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage")
    }
    print(
        f"  Avg FID={averages['FID']:.4f} | "
        f"IS={averages['IS_mean']:.4f}±{averages['IS_std']:.4f} | "
        f"P={averages['precision']:.4f} | R={averages['recall']:.4f}"
    )
    return {
        "step": step,
        "ckpt_name": Path(ckpt_path).name,
        **metrics,
        "avg_FID": averages["FID"],
        "avg_IS_mean": averages["IS_mean"],
        "avg_IS_std": averages["IS_std"],
        "avg_precision": averages["precision"],
        "avg_recall": averages["recall"],
        "avg_density": averages["density"],
        "avg_coverage": averages["coverage"],
    }

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def remove_generated_duplicates_of_reused_images(final_dir, reused_prefix):
    final_dir = Path(final_dir)
    reused_hashes = {file_sha256(path) for path in final_dir.glob(f"{reused_prefix}_*.png")}
    removed = []
    for path in sorted(final_dir.glob("gen_*.png")):
        if file_sha256(path) in reused_hashes:
            path.unlink()
            removed.append(path)
    if removed:
        print(f"  Removed {len(removed)} gen_* copies overlapping reused images.")
    return removed

def duplicate_png_groups(directory):
    by_hash = {}
    for path in sorted(Path(directory).glob("*.png")):
        if path.name.startswith(".tmp_"):
            continue
        by_hash.setdefault(file_sha256(path), []).append(path.name)
    return {digest: names for digest, names in by_hash.items() if len(names) > 1}

In [ ]:
# Evaluate every checkpoint using validation data only
checkpoints = discover_checkpoints(SD_OUTPUT_DIR)
print(f"Found {len(checkpoints)} checkpoints:")
for _step, _path in checkpoints:
    print(f"  checkpoint-{_step}")

# Workers complete generation before metrics; JSON and caches remain in the notebook process.
if RUN_EVALUATION_PHASE:
    _sd_parallel_generate_checkpoint_jobs(
        checkpoints, LOCAL_MODEL_DIR, EVAL_DIR, NEGATIVE_PROMPT, POSITIVE_PROMPT,
        N_EVAL_IMAGES_PER_CLASS, INFERENCE_STEPS, EVAL_GUIDANCE_SCALE, EVAL_SEED, RESOLUTION,
    )

from parallel_generation_utils import checkpoint_content_signature, file_content_signature, png_content_signature, sd_metrics_cache_config, sd_metrics_cache_compatible
CHECKPOINT_EVAL_CACHE_PATH = EVAL_METRICS_PATH.with_name("checkpoint_validation_cache_v2.json")
checkpoint_eval_config = sd_metrics_cache_config(eval_seed=EVAL_SEED, inference_steps=INFERENCE_STEPS, guidance_scale=EVAL_GUIDANCE_SCALE, resolution=RESOLUTION, n_gen_per_class=N_EVAL_IMAGES_PER_CLASS, checkpoint_type=SD_CHECKPOINT_TYPE, base_model_dir=LOCAL_MODEL_DIR, n_validation_images_per_class=N_VALIDATION_IMAGES_PER_CLASS, prdc_nearest_k=PRDC_NEAREST_K, evaluator_batch_size=8, evaluator_num_workers=0, metric_backend="GenerativeEvaluator", metric_backend_version="torchmetrics_fid_is_prdc_v1")
validation_csv_signature = file_content_signature(VALIDATION_METADATA_PATH)
current_checkpoint_inputs = {
    path.name: {
        "checkpoint_signature": checkpoint_content_signature(path),
        "negative_image_signature": png_content_signature(EVAL_DIR / path.name / "negative"),
        "positive_image_signature": png_content_signature(EVAL_DIR / path.name / "positive"),
    } for _, path in checkpoints
}
all_metrics = None
if CHECKPOINT_EVAL_CACHE_PATH.is_file():
    with CHECKPOINT_EVAL_CACHE_PATH.open(encoding="utf-8") as handle:
        cache_payload = json.load(handle)
    cached_entries = cache_payload.get("checkpoints", {}) if sd_metrics_cache_compatible(cache_payload, checkpoint_eval_config, VALIDATION_METADATA_PATH) else {}
    if set(cached_entries) == set(current_checkpoint_inputs) and all(all(cached_entries[name].get(key) == value for key, value in signature.items()) for name, signature in current_checkpoint_inputs.items()):
        all_metrics = [cached_entries[path.name]["metrics"] for _, path in checkpoints]
        print("Validation metrics loaded from a v2 cache with matching signatures.")

if all_metrics is None and not RUN_EVALUATION_PHASE:
    raise RuntimeError(
        "Evaluation is disabled, but the content-aware checkpoint-metrics cache is missing or incompatible."
    )

if all_metrics is None:
    all_metrics = []
    with temporary_real_reference_dirs_from_split_metadata(
        metadata_path=VALIDATION_METADATA_PATH,
        n_per_class=N_VALIDATION_IMAGES_PER_CLASS,
        seed=EVAL_SEED,
    ) as (real_neg_ref_dir, real_pos_ref_dir):
        print("\nDirectory temporanee validation:")
        print("  Negative:", real_neg_ref_dir)
        print("  Positive:", real_pos_ref_dir)

        with measure_sustainability(label="checkpoint_validation_evaluation", sample_interval=0.5) as eco_eval:
            for step, ckpt_path in checkpoints:
                all_metrics.append(eval_one_checkpoint(
                    step=step,
                    ckpt_path=ckpt_path,
                    base_model_dir=LOCAL_MODEL_DIR,
                    real_neg_dir=real_neg_ref_dir,
                    real_pos_dir=real_pos_ref_dir,
                    eval_dir=EVAL_DIR,
                    n_gen=N_EVAL_IMAGES_PER_CLASS,
                    inference_steps=INFERENCE_STEPS,
                    guidance_scale=EVAL_GUIDANCE_SCALE,
                    seed=EVAL_SEED,
                    resolution=RESOLUTION,
                ))

    with EVAL_METRICS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(all_metrics, handle, indent=2, ensure_ascii=False)
    with CHECKPOINT_EVAL_CACHE_PATH.open("w", encoding="utf-8") as handle:
        json.dump({"schema_version": 2, "config": checkpoint_eval_config, "validation_csv_signature": validation_csv_signature, "checkpoints": {path.name: {**current_checkpoint_inputs[path.name], "metrics": metrics} for (_, path), metrics in zip(checkpoints, all_metrics)}}, handle, indent=2, ensure_ascii=False)

    eco_record = eco_eval.metrics.to_dict()
    eco_record.update({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "record_type": "checkpoint_validation_evaluation",
        "n_checkpoints": len(checkpoints),
        "n_generated_images_per_class": N_EVAL_IMAGES_PER_CLASS,
        "n_real_images_per_class": N_VALIDATION_IMAGES_PER_CLASS,
        "real_reference_metadata": str(VALIDATION_METADATA_PATH),
    })
    with EVAL_SUSTAINABILITY_LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(eco_record, ensure_ascii=False) + "\n")

    print(f"\nMetriche eco-tracking validation:\n{eco_eval.metrics}")
    print(f"Metriche validation salvate in: {EVAL_METRICS_PATH}")

In [ ]:
# Select the best checkpoint
# Reload from file (idempotent when the cell is rerun)
with open(EVAL_METRICS_PATH, encoding="utf-8") as f:
    all_metrics = json.load(f)

df_eval = pd.DataFrame([
    {
        "checkpoint":    m["ckpt_name"],
        "step":          m["step"],
        "avg_FID":       m["avg_FID"],
        "avg_IS_mean":   m["avg_IS_mean"],
        "avg_IS_std":    m["avg_IS_std"],
        "avg_precision": m["avg_precision"],
        "avg_recall":    m["avg_recall"],
        "avg_density":   m["avg_density"],
        "avg_coverage":  m["avg_coverage"],
        "FID_neg":       m["negative"]["FID"],
        "FID_pos":       m["positive"]["FID"],
        "IS_neg":        m["negative"]["IS_mean"],
        "IS_pos":        m["positive"]["IS_mean"],
        "precision_neg": m["negative"]["precision"],
        "precision_pos": m["positive"]["precision"],
        "recall_neg":    m["negative"]["recall"],
        "recall_pos":    m["positive"]["recall"],
    }
    for m in all_metrics
]).sort_values("avg_FID").reset_index(drop=True)

print("Metric summary (sorted by increasing avg_FID, best first):")
print(df_eval.to_string(index=False))

best_row = df_eval.iloc[0]
BEST_CHECKPOINT = SD_OUTPUT_DIR / best_row["checkpoint"]

print(f"\nBest checkpoint : {best_row['checkpoint']}")
print(f"  avg FID       : {best_row['avg_FID']:.4f}")
print(f"  avg IS        : {best_row['avg_IS_mean']:.4f} ± {best_row['avg_IS_std']:.4f}")
print(f"  avg precision : {best_row['avg_precision']:.4f}")
print(f"  avg recall    : {best_row['avg_recall']:.4f}")
print(f"  Path          : {BEST_CHECKPOINT}")

# Keep diagnostic plots separate; FID remains in the next cell's dedicated figure
df_eval_by_step = df_eval.sort_values("step").reset_index(drop=True)
best_step = int(best_row["step"])

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
axis.plot(
    df_eval_by_step["step"],
    df_eval_by_step["avg_IS_mean"],
    color="#9467bd",
    marker="o",
    label="Mean IS",
)
axis.fill_between(
    df_eval_by_step["step"],
    df_eval_by_step["avg_IS_mean"] - df_eval_by_step["avg_IS_std"],
    df_eval_by_step["avg_IS_mean"] + df_eval_by_step["avg_IS_std"],
    color="#9467bd",
    alpha=0.2,
    label="± standard deviation",
)
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(title="Mean Inception Score per checkpoint", xlabel="Training step", ylabel="IS")
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "is_per_checkpoint.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
for column, label, color in [
    ("avg_precision", "Mean precision", "#1f77b4"),
    ("avg_recall", "Mean recall", "#ff7f0e"),
]:
    axis.plot(
        df_eval_by_step["step"],
        df_eval_by_step[column],
        color=color,
        marker="o",
        label=label,
    )
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(
    title="Mean precision and recall by checkpoint",
    xlabel="Training step",
    ylabel="PRDC value",
)
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "precision_recall_over_steps.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
for column, label, color in [
    ("avg_density", "Mean density", "#2ca02c"),
    ("avg_coverage", "Mean coverage", "#d62728"),
]:
    axis.plot(
        df_eval_by_step["step"],
        df_eval_by_step[column],
        color=color,
        marker="o",
        label=label,
    )
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(
    title="Mean density and coverage by checkpoint",
    xlabel="Training step",
    ylabel="PRDC value",
)
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "density_coverage_over_steps.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

print("Separate checkpoint diagnostic plots saved under:", PLOTS_DIR)

### Checkpoint-selection diagnostics

These read-only figures convert the saved validation JSON into separate views of mean FID,
Inception Score, precision/recall, density/coverage, and the precision-recall plane across
training steps. The selected checkpoint is highlighted solely to visualize the predeclared
minimum-mean-FID rule; no figure changes that rule or recomputes an embedding.

Figures are saved under `results/2_diffusers/02_sd21_filtered_100steps/plots/`. Trends should
be interpreted jointly and cautiously because the validation reference is small and the
metrics capture different fidelity/coverage properties.

In [ ]:
with EVAL_METRICS_PATH.open(encoding="utf-8") as handle:
    checkpoint_plot_metrics = json.load(handle)

df_checkpoint_plots = pd.DataFrame([
    {
        "checkpoint": row["ckpt_name"],
        "step": row["step"],
        "FID negative": row["negative"]["FID"],
        "FID positive": row["positive"]["FID"],
        "Mean FID": row["avg_FID"],
        "mean precision": row["avg_precision"],
        "mean recall": row["avg_recall"],
    }
    for row in checkpoint_plot_metrics
]).sort_values("step")

best_plot_row = df_checkpoint_plots.loc[df_checkpoint_plots["Mean FID"].idxmin()]

fig, ax = plt.subplots(figsize=(10, 5))
for column, marker in [
    ("FID negative", "o"),
    ("FID positive", "s"),
    ("Mean FID", "^"),
]:
    ax.plot(df_checkpoint_plots["step"], df_checkpoint_plots[column], marker=marker, label=column)
ax.scatter(
    [best_plot_row["step"]],
    [best_plot_row["Mean FID"]],
    s=180,
    facecolors="none",
    edgecolors="red",
    linewidths=2,
    label=f"best: {best_plot_row['checkpoint']}",
)
ax.set(title="Validation-set FID by checkpoint", xlabel="Training step", ylabel="FID")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "fid_per_checkpoint.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    df_checkpoint_plots["mean recall"],
    df_checkpoint_plots["mean precision"],
    c=df_checkpoint_plots["step"],
    cmap="viridis",
    s=65,
)
ax.scatter(
    [best_plot_row["mean recall"]],
    [best_plot_row["mean precision"]],
    s=220,
    facecolors="none",
    edgecolors="red",
    linewidths=2,
    label=f"best FID: {best_plot_row['checkpoint']}",
)
ax.set(
    title="Mean PRDC per checkpoint",
    xlabel="Mean recall",
    ylabel="Mean precision",
)
ax.grid(alpha=0.25)
ax.legend()
fig.colorbar(scatter, ax=ax, label="Training step")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "precision_recall_per_checkpoint.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Grafici checkpoint salvati in:", PLOTS_DIR)

## 6. Final 100-step RAW generation

The validation-selected checkpoint generates a resumable RAW pool of exactly 2,722 images
for each class. The 100 class-specific samples already produced for that checkpoint are
reused as synthetic inputs, and only missing deterministic indices are generated. Runtime
GPU discovery and reservation-aware scheduling permit one or more available devices without
encoding a machine-specific identifier.

A complete local RAW pool is reused when the runtime and final-generation plans confirm valid
filenames, readable images, the exact target count, and unique content. The per-directory
generation manifest and checkpoint signature are validated only when missing images must be
scheduled. When `generation_info.json` exists, a mismatch with the validation-selected checkpoint
stops execution; historical fingerprint mismatches do not invalidate an otherwise complete local
pool. Generation state and runtime telemetry remain recorded in the G02 results namespace.

In [ ]:
# Final generation with the best checkpoint
print(f"Best checkpoint  : {BEST_CHECKPOINT.name}")
print(f"Images/class  : {N_FINAL_IMAGES_PER_CLASS}")
print(f"Output           : {FINAL_GEN_DIR}")

BEST_EVAL_DIR = EVAL_DIR / BEST_CHECKPOINT.name
_class_config = {
    "negative": {
        "prompt": NEGATIVE_PROMPT,
        "final_dir": FINAL_NEG_DIR,
        "eval_dir": BEST_EVAL_DIR / "negative",
        "seed": EVAL_SEED,
        "prefix": "eval_neg",
    },
    "positive": {
        "prompt": POSITIVE_PROMPT,
        "final_dir": FINAL_POS_DIR,
        "eval_dir": BEST_EVAL_DIR / "positive",
        "seed": EVAL_SEED + 1,
        "prefix": "eval_pos",
    },
}

from parallel_generation_utils import SD_SEED_STRATEGY, copy_validated_sd_evaluation_images, final_sd_generation_plan

def _final_plan(cfg):
    return final_sd_generation_plan(Path(cfg["final_dir"]), N_FINAL_IMAGES_PER_CLASS, cfg["prefix"])

_final_already_complete = all(
    _final_plan(_class_config[_cls])["complete"]
    for _cls in FINAL_GENERATE_CLASSES
)
if _final_already_complete:
    print("Final PNG set complete; still validating files and manifest parameters before skipping.")
generation_info_path = GENERATION_INFO_PATH
if generation_info_path.exists():
    with open(generation_info_path, encoding="utf-8") as f:
        previous_generation = json.load(f)
    previous_best = previous_generation.get("best_checkpoint")
    final_images_exist = any(any(Path(cfg["final_dir"]).glob("*.png")) for cfg in _class_config.values())
    if previous_best and previous_best != BEST_CHECKPOINT.name and final_images_exist:
        raise RuntimeError(
            f"The best checkpoint changed from {previous_best} to {BEST_CHECKPOINT.name}. "
            "Use a new FINAL_GEN_DIR or deliberately clear the final directories."
        )

removed_overlaps = {}

with measure_sustainability(label=f"final_gen_{BEST_CHECKPOINT.name}", sample_interval=0.5) as eco_final:
    for _cls in FINAL_GENERATE_CLASSES:
        cfg = _class_config[_cls]
        eval_available = N_EVAL_IMAGES_PER_CLASS

        print(f"\nClass {_cls.upper()}")
        print(f"  Final target              : {N_FINAL_IMAGES_PER_CLASS}")
        print(f"  Reused evaluation images : {eval_available}")

        if RUN_GENERATION_PHASE:
            # Copy reused images first, then complete the total.
            reuse_record = copy_validated_sd_evaluation_images(
                source_dir=cfg["eval_dir"], final_dir=cfg["final_dir"], count=eval_available,
                reused_prefix=cfg["prefix"], checkpoint_path=str(BEST_CHECKPOINT),
                class_name=_cls, prompt=cfg["prompt"],
                base_seed=EVAL_SEED, num_inference_steps=INFERENCE_STEPS,
                guidance_scale=EVAL_GUIDANCE_SCALE, resolution=RESOLUTION,
                checkpoint_type=SD_CHECKPOINT_TYPE, base_model_dir=LOCAL_MODEL_DIR,
            )
            n_reused = len(reuse_record["files"])
            n_new_target = N_FINAL_IMAGES_PER_CLASS - n_reused
            print(f"  Actually copied from evaluation: {n_reused}")
            print(f"  gen_* share of the final dataset : {n_new_target}")

        else:
            _existing_plan = _final_plan(cfg)
            if not _existing_plan["complete"]:
                raise RuntimeError(
                    f"Generation is disabled, but the existing {_cls} pool is incomplete: {_existing_plan}"
                )
            n_reused = _existing_plan["n_valid_reused"]
            n_new_target = N_FINAL_IMAGES_PER_CLASS - n_reused
            print(f"  Existing complete pool validated; reused-prefix images: {n_reused}")

        if RUN_GENERATION_PHASE:
            removed = remove_generated_duplicates_of_reused_images(
                final_dir=cfg["final_dir"],
                reused_prefix=cfg["prefix"],
            )
            removed_overlaps[_cls] = len(removed)

        else:
            removed = []

        _plan_before = _final_plan(cfg)
        n_before = _plan_before["n_valid_reused"] + len(_plan_before["valid_gen_indices"])
        print(f"  Unique images present   : {n_before}")
        print(f"  New images to generate          : {max(0, N_FINAL_IMAGES_PER_CLASS - n_before)}")

        if RUN_GENERATION_PHASE:
            _sd_parallel_generate_final(
                BEST_CHECKPOINT, LOCAL_MODEL_DIR, cfg["prompt"], cfg["final_dir"], N_FINAL_IMAGES_PER_CLASS,
                INFERENCE_STEPS, EVAL_GUIDANCE_SCALE, cfg["seed"], RESOLUTION, _cls, cfg["prefix"], n_reused,
            )

        duplicates = duplicate_png_groups(cfg["final_dir"])
        if duplicates:
            raise RuntimeError(f"Found {len(duplicates)} duplicate groups under {cfg['final_dir']}.")
        _verified_plan = _final_plan(cfg)
        if not _verified_plan["complete"]:
            raise RuntimeError(f"Invalid final dataset per {_cls}: {_verified_plan}")

gc.collect()
torch.cuda.empty_cache()

_n_neg = N_FINAL_IMAGES_PER_CLASS if _final_plan(_class_config["negative"])["complete"] else 0
_n_pos = N_FINAL_IMAGES_PER_CLASS if _final_plan(_class_config["positive"])["complete"] else 0
_final_record = eco_final.metrics.to_dict()
_final_record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "record_type": "final_generation",
    "generation_log_schema": 2,
    "best_checkpoint": BEST_CHECKPOINT.name,
    "selection_reference": "validation",
    "selection_metrics_path": str(EVAL_METRICS_PATH),
    "avg_FID": float(best_row["avg_FID"]),
    "avg_IS_mean": float(best_row["avg_IS_mean"]),
    "n_per_class": N_FINAL_IMAGES_PER_CLASS,
    "n_eval_reused": N_EVAL_IMAGES_PER_CLASS,
    "removed_overlaps": removed_overlaps,
    "n_negative_final": _n_neg,
    "n_positive_final": _n_pos,
    "inference_steps": INFERENCE_STEPS,
    "guidance_scale": EVAL_GUIDANCE_SCALE,
    "generated_classes": FINAL_GENERATE_CLASSES,
    "generated_negative": str(FINAL_NEG_DIR),
    "generated_positive": str(FINAL_POS_DIR),
    "status": "completed",
    "seed_strategy": SD_SEED_STRATEGY,
    "parallel_generation": PARALLEL_GENERATION,
    "generation_gpu_devices": GENERATION_GPU_DEVICES,
})

if RUN_GENERATION_PHASE:
    with open(generation_info_path, "w", encoding="utf-8") as f:
        json.dump(_final_record, f, indent=2, ensure_ascii=False)
    with open(FINAL_GENERATION_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(_final_record, ensure_ascii=False) + "\n")
else:
    print("Generation phase skipped; existing pool and checkpoint compatibility were validated without mutation.")

print(f"\nFinal-generation EcoTracker metrics:\n{eco_final.metrics}")
print(f"Positive finali: {_n_pos} -> {FINAL_POS_DIR}")

### 6.1 Deterministic size-matched RAW reference

A direct filter comparison requires equal sample counts. For each class, this section draws
1,361 images without replacement from the complete 2,722-image RAW pool after sorting the
source filenames. A class-specific deterministic seed derived from `RAW_MATCHED_SEED` makes
the selection reproducible.

The selected copies are stored under `generated_images/raw_matched_1361/`, with a manifest
that binds the source directory, selection method, seed, filenames, and expected count.
Reuse is allowed only when the manifest and source pool remain compatible; otherwise the
matched set is rebuilt or the validation fails. This pool is a random size-matched baseline,
not an additional generator.

In [ ]:
# Build the matched RAW dataset at 100 inference steps.
RAW_MATCHED_SELECTION_METHOD = "sorted_png_without_replacement_numpy_default_rng_v1"
RAW_COMPLETE_COUNT_PER_CLASS = N_FINAL_IMAGES_PER_CLASS

def validate_png_file(path):
    path = Path(path)
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as exc:
        raise RuntimeError(f"Invalid PNG: {path}") from exc
    return True

def png_names_digest(paths):
    digest = hashlib.sha256()
    for path in paths:
        digest.update(path.name.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

def deterministic_raw_matched_selection(raw_dir, label):
    raw_dir = Path(raw_dir)
    from parallel_generation_utils import GENERATED_PNG_PATTERN, readable_png_paths
    raw_paths = readable_png_paths(raw_dir, GENERATED_PNG_PATTERN)
    if len(raw_paths) != RAW_COMPLETE_COUNT_PER_CLASS:
        raise RuntimeError(
            f"Inconsistent completed RAW set in {raw_dir}: "
            f"{len(raw_paths)} != {RAW_COMPLETE_COUNT_PER_CLASS}."
        )
    for path in raw_paths:
        validate_png_file(path)

    random_seed = RAW_MATCHED_SEED + int(label)
    rng = np.random.default_rng(random_seed)
    selected_indices = sorted(
        int(index)
        for index in rng.choice(len(raw_paths), size=RAW_MATCHED_COUNT, replace=False)
    )
    selected_paths = [raw_paths[index] for index in selected_indices]
    selected_names = [path.name for path in selected_paths]
    if len(selected_names) != len(set(selected_names)):
        raise RuntimeError("The matched RAW selection contains duplicate filenames.")
    return raw_paths, selected_paths, selected_names, random_seed

def load_json_if_present(path):
    path = Path(path)
    if not path.is_file():
        return None
    try:
        with path.open(encoding="utf-8") as handle:
            return json.load(handle)
    except (OSError, json.JSONDecodeError):
        return None

def raw_matched_manifest_compatible(payload, class_name, label, raw_dir, output_dir, raw_paths, selected_names, random_seed):
    if not payload:
        return False
    expected = {
        "schema_version": 1,
        "experiment_name": EXPERIMENT_NAME,
        "class_name": class_name,
        "label": int(label),
        "source_raw_dir": str(Path(raw_dir)),
        "output_dir": str(Path(output_dir)),
        "n_raw_available": RAW_COMPLETE_COUNT_PER_CLASS,
        "n_selected": RAW_MATCHED_COUNT,
        "random_seed": int(random_seed),
        "selection_method": RAW_MATCHED_SELECTION_METHOD,
        "source_raw_names_sha256": png_names_digest(raw_paths),
    }
    for key, expected_value in expected.items():
        if payload.get(key) != expected_value:
            return False
    return payload.get("selected_names") == selected_names

def ensure_raw_matched_dataset(class_name):
    label = FINAL_CLASS_LABELS[class_name]
    raw_dir = FINAL_DIRS[class_name]
    output_dir = RAW_MATCHED_DIRS[class_name]
    manifest_path = RAW_MATCHED_MANIFEST_PATHS[class_name]
    raw_paths, selected_paths, selected_names, random_seed = deterministic_raw_matched_selection(
        raw_dir=raw_dir,
        label=label,
    )

    payload = load_json_if_present(manifest_path)
    output_paths = [output_dir / name for name in selected_names]
    can_skip = (
        raw_matched_manifest_compatible(
            payload, class_name, label, raw_dir, output_dir, raw_paths, selected_names, random_seed
        )
        and count_pngs(output_dir) == RAW_MATCHED_COUNT
        and all(path.is_file() for path in output_paths)
        and not duplicate_png_groups(output_dir)
    )
    if can_skip:
        for path in output_paths:
            validate_png_file(path)
        print(f"RAW matched {class_name}: compatible manifest and files validated; reuse")
        return payload

    if not RUN_GENERATION_PHASE:
        raise RuntimeError(
            f"Generation is disabled, but the RAW-matched {class_name} artifact is missing or incompatible."
        )

    output_dir.mkdir(parents=True, exist_ok=True)
    for path in output_dir.glob("*.png"):
        path.unlink()
    for source_path in selected_paths:
        shutil.copy2(source_path, output_dir / source_path.name)

    if count_pngs(output_dir) != RAW_MATCHED_COUNT:
        raise RuntimeError(f"Incomplete matched RAW set for {class_name}: {output_dir}")
    if duplicate_png_groups(output_dir):
        raise RuntimeError(f"Matched RAW set {class_name} contains byte-identical duplicates.")
    for path in output_paths:
        validate_png_file(path)

    manifest = {
        "schema_version": 1,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "experiment_name": EXPERIMENT_NAME,
        "class_name": class_name,
        "label": int(label),
        "source_raw_dir": str(raw_dir),
        "output_dir": str(output_dir),
        "n_raw_available": len(raw_paths),
        "n_selected": len(selected_paths),
        "random_seed": int(random_seed),
        "selection_method": RAW_MATCHED_SELECTION_METHOD,
        "source_raw_names_sha256": png_names_digest(raw_paths),
        "selected_names": selected_names,
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
    print(f"RAW matched {class_name}: create {len(selected_paths)} images -> {output_dir}")
    print("Manifest:", manifest_path)
    return manifest

for class_name in FINAL_GENERATE_CLASSES:
    ensure_raw_matched_dataset(class_name)

## 7. Class-conditional adaptive filtering

The shared adaptive filter is calibrated independently for negative and positive images
using only real training examples of the corresponding class. It characterizes foreground
proportion, tissue intensity mean and dispersion, entropy, Laplacian variance, and main-mask
compactness. Each RAW synthetic image is subjected to morphological and quality checks, then
scored by robust distance from the real-training reference statistics.

Accepted candidates are ranked by score and the top 1,361 per class are exported to
`data/synthetic/02_sd21_filtered_100steps/`. Per-image CSV reports and class-level JSON
summaries under `results/.../metrics/` record decisions and rejection reasons. Existing
outputs are reused unless `RUN_FILTER_PHASE` asks for the filter to run again. Because the filter is
trained-data-calibrated selection, it may improve apparent fidelity while reducing support or
diversity; the next validation comparison measures that trade-off rather than assuming a
uniformly beneficial effect.

In [ ]:
def file_signature(path):
    path = Path(path)
    return {
        "path": str(path),
        "size": path.stat().st_size,
        "mtime_ns": path.stat().st_mtime_ns,
    }

def png_files_signature(directory):
    return [
        {
            "name": path.name,
            "size": path.stat().st_size,
            "mtime_ns": path.stat().st_mtime_ns,
        }
        for path in sorted(Path(directory).glob("*.png"))
    ]

def evaluate_generated_dir_against_split(generated_dir, metadata_path, label, n_images, seed):
    generated_dir = Path(generated_dir)
    n_generated = count_pngs(generated_dir)
    if n_generated == 0:
        raise RuntimeError(f"No generated images under {generated_dir}.")
    duplicates = duplicate_png_groups(generated_dir)
    if duplicates:
        raise RuntimeError(f"Found {len(duplicates)} duplicate groups under {generated_dir}.")

    with temporary_real_reference_dir_from_split_metadata(
        metadata_path=metadata_path,
        label=label,
        n_images=n_images,
        seed=seed,
    ) as real_reference_dir:
        evaluator = GenerativeEvaluator(
            real_dir=real_reference_dir,
            generated_dir=generated_dir,
            batch_size=8,
            num_workers=0,
        )
        fid_is_metrics, real_features, fake_features = evaluator.compute_with_features()
        prdc_metrics = compute_prdc_metrics(real_features, fake_features)

    return {
        **fid_is_metrics,
        **prdc_metrics,
        "n_generated": n_generated,
    }

In [ ]:
# IDEMPOTENT_GUARD_V1:filter
filter_summaries = {}

if RUN_FILTER_PHASE:
    filter_script = UTILITY_DIR / "run_adaptive_filter.py"
    if not filter_script.is_file():
        raise FileNotFoundError(f"Adaptive-filter script not found: {filter_script}")

    # Calibrate and execute the filter independently for each target class.
    for class_name in FINAL_GENERATE_CLASSES:
        class_label = FINAL_CLASS_LABELS[class_name]
        command = [
            sys.executable,
            str(filter_script),
            "--class-name", class_name,
            "--label", str(class_label),
            "--raw-dir", str(FINAL_DIRS[class_name]),
            "--filtered-dir", str(FILTERED_DIRS[class_name]),
            "--n-selected", str(N_SELECTED_PER_CLASS),
            "--train-metadata", str(TRAIN_METADATA_PATH),
            "--data-processed-dir", str(DATA_PROCESSED_DIR),
            "--report-csv", str(FILTER_REPORT_PATHS[class_name]),
            "--summary-json", str(FILTER_SUMMARY_PATHS[class_name]),
            "--experiment-name", EXPERIMENT_NAME,
            "--eval-seed", str(EVAL_SEED),
            "--nonblack-threshold", str(NONBLACK_THRESHOLD),
        ]
        subprocess.run(command, check=True)

# Whether the phase ran or was skipped, require complete, reusable filter evidence.
for class_name in FINAL_GENERATE_CLASSES:
    filtered_dir = FILTERED_DIRS[class_name]
    report_path = FILTER_REPORT_PATHS[class_name]
    summary_path = FILTER_SUMMARY_PATHS[class_name]
    if not summary_path.is_file():
        raise FileNotFoundError(f"Filter summary not found for {class_name}: {summary_path}")
    if not report_path.is_file():
        raise FileNotFoundError(f"Filter report not found for {class_name}: {report_path}")
    with summary_path.open("r", encoding="utf-8") as handle:
        filter_summaries[class_name] = json.load(handle)
    if count_pngs(filtered_dir) != N_SELECTED_PER_CLASS:
        raise RuntimeError(
            f"Invalid filtered-image count for {class_name}: "
            f"{count_pngs(filtered_dir)} != {N_SELECTED_PER_CLASS}."
        )
    if duplicate_png_groups(filtered_dir):
        raise RuntimeError(f"The filtered {class_name} dataset contains byte-identical duplicates.")

print("\nAdaptive-filter summaries:")
for class_name, summary in filter_summaries.items():
    print(f"{class_name}: {summary.get('status', 'unknown')} -> {FILTERED_DIRS[class_name]}")

## 8. Validation comparison of RAW and filtered pools

This section evaluates three 100-step representations against the same validation split:
the complete RAW pool (`n=2,722` per class), the deterministic RAW matched pool (`n=1,361`),
and the filtered pool (`n=1,361`). The direct size-controlled descriptive contrast for the selection step is RAW matched
versus filtered because both contain the same number of images; the complete RAW pool is
retained only as descriptive context. This comparison does not, by itself, identify a
causal filtering effect because membership differs between the two pools.

FID, Inception Score, precision, recall, density, and coverage are written to
`validation_comparison_100_steps_raw_matched_vs_filtered.csv` and its JSON companion. The
test split is not accessed. Lower FID and higher precision/density may indicate improved
fidelity, whereas recall/coverage describe retained support; no single metric establishes
clinical realism. With 73 real references per class and a general-domain Inception feature
space, both absolute values and small deltas require caution. Cached results are reused only
when compatible with the declared comparison.

In [ ]:
metric_columns = [
    "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage"
]
validation_sets = {}
for class_name in FINAL_GENERATE_CLASSES:
    label = FINAL_CLASS_LABELS[class_name]
    validation_sets[f"{class_name}_raw_complete_{N_FINAL_IMAGES_PER_CLASS}"] = {
        "class": class_name,
        "label": label,
        "stage": "raw",
        "comparison_role": "context_only",
        "generated_dir": FINAL_DIRS[class_name],
        "expected_size": N_FINAL_IMAGES_PER_CLASS,
    }
    validation_sets[f"{class_name}_raw_matched_{RAW_MATCHED_COUNT}"] = {
        "class": class_name,
        "label": label,
        "stage": "raw_matched",
        "comparison_role": "filter_comparison",
        "generated_dir": RAW_MATCHED_DIRS[class_name],
        "expected_size": RAW_MATCHED_COUNT,
    }
    validation_sets[f"{class_name}_filtered_{N_SELECTED_PER_CLASS}_adaptive_mask"] = {
        "class": class_name,
        "label": label,
        "stage": "filtered",
        "comparison_role": "filter_comparison",
        "generated_dir": FILTERED_DIRS[class_name],
        "expected_size": N_SELECTED_PER_CLASS,
    }
validation_eval_config = {
    "schema_version": 2,
    "metric_backend": "generative_evaluator.py",
    "reference_split": "validation",
    "n_validation_per_class": N_VALIDATION_IMAGES_PER_CLASS,
    "knn_k": PRDC_NEAREST_K,
    "inception_batch": 8,
    "classes": FINAL_GENERATE_CLASSES,
    "sets": {
        set_name: {
            "class": spec["class"],
            "label": int(spec["label"]),
            "stage": spec["stage"],
            "comparison_role": spec["comparison_role"],
            "expected_size": int(spec["expected_size"]),
        }
        for set_name, spec in validation_sets.items()
    },
}
validation_input_signature = {
    "sets": {
        set_name: png_files_signature(spec["generated_dir"])
        for set_name, spec in validation_sets.items()
    },
    "raw_matched_manifests": {
        class_name: file_signature(path) if Path(path).is_file() else None
        for class_name, path in RAW_MATCHED_MANIFEST_PATHS.items()
    },
    "validation_csv": file_signature(VALIDATION_METADATA_PATH),
}

use_validation_cache = False
if VALIDATION_COMPARISON_JSON.exists() and not FORCE_RECOMPUTE_VALIDATION_COMPARISON:
    try:
        with VALIDATION_COMPARISON_JSON.open("r", encoding="utf-8") as handle:
            validation_payload = json.load(handle)
        use_validation_cache = (
            validation_payload.get("schema_version") == 2
            and validation_payload.get("config") == validation_eval_config
            and validation_payload.get("input_signature") == validation_input_signature
            and VALIDATION_COMPARISON_CSV.exists()
        )
    except Exception as exc:
        print(f"Unreadable validation cache; recomputing: {exc}")

if not RUN_EVALUATION_PHASE and not use_validation_cache:
    raise RuntimeError(
        "Evaluation is disabled, but no compatible validation comparison cache is available."
    )

if use_validation_cache:
    df_validation_comparison = pd.read_csv(VALIDATION_COMPARISON_CSV)
    print("Metriche validation caricate da cache:", VALIDATION_COMPARISON_JSON)
else:
    comparison_rows = []
    for set_name, spec in validation_sets.items():
        class_name = spec["class"]
        label = spec["label"]
        generated_dir = spec["generated_dir"]
        expected_size = spec["expected_size"]
        n_generated = count_pngs(generated_dir)
        if n_generated != expected_size:
            raise RuntimeError(
                f"Incomplete {set_name} dataset: {n_generated} != {expected_size}."
            )
        print(f"Validation evaluation: {set_name}")
        metrics = evaluate_generated_dir_against_split(
            generated_dir=generated_dir,
            metadata_path=VALIDATION_METADATA_PATH,
            label=label,
            n_images=N_VALIDATION_IMAGES_PER_CLASS,
            seed=EVAL_SEED + label,
        )
        comparison_rows.append({
            "class": class_name,
            "label": int(label),
            "set_name": set_name,
            "stage": spec["stage"],
            "comparison_role": spec["comparison_role"],
            **metrics,
        })

    df_validation_comparison = pd.DataFrame(comparison_rows)[
        [
            "class", "label", "set_name", "stage", "comparison_role",
            *metric_columns, "n_generated",
        ]
    ]
    df_validation_comparison.to_csv(VALIDATION_COMPARISON_CSV, index=False)
    with VALIDATION_COMPARISON_JSON.open("w", encoding="utf-8") as handle:
        json.dump({
            "schema_version": 2,
            "config": validation_eval_config,
            "input_signature": validation_input_signature,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "experiment": EXPERIMENT_NAME,
            "evaluation_reference": "validation",
            "metrics": df_validation_comparison.to_dict(orient="records"),
        }, handle, indent=2, ensure_ascii=False)

print("\nConfronto a 100 step su validation:")
print(df_validation_comparison.to_string(index=False))
print("\nSaved in:", VALIDATION_COMPARISON_CSV)

### Qualitative review of filtered samples

Deterministically selected grids display negative and positive filtered samples for visual
inspection of orientation, foreground extent, contrast, and obvious artifacts. The cell reads
the exported pool and does not modify filtering decisions or recompute quantitative metrics.

These grids are diagnostic evidence only: a small curated view cannot establish diversity,
absence of memorization, or diagnostic validity. The quantitative validation comparison and
the later unified benchmark provide the corresponding distribution-level analyses.

In [ ]:
for class_name in FINAL_GENERATE_CLASSES:
    sample_dir = FILTERED_DIRS[class_name]
    image_paths = sorted(sample_dir.glob("*.png"))
    if not image_paths:
        print(f"No images available for the grid {class_name}.")
        continue

    n_samples = min(16, len(image_paths))
    sample_indices = np.linspace(0, len(image_paths) - 1, n_samples, dtype=int)
    selected_paths = [image_paths[index] for index in sample_indices]

    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for ax, image_path in zip(axes.flat, selected_paths):
        with Image.open(image_path) as image:
            ax.imshow(image.convert("L"), cmap="gray")
        ax.set_title(image_path.name, fontsize=7)
        ax.axis("off")
    for ax in axes.flat[n_samples:]:
        ax.axis("off")
    fig.suptitle(f"Filtered generated samples - {class_name}")
    fig.tight_layout()
    output_path = PLOTS_DIR / f"samples_{class_name}.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Filtered grids saved under:", PLOTS_DIR)

### Checkpoint sample pages

The following cells read previously generated checkpoint samples and create paginated grids
with at most six checkpoints per page. No image generation, embedding extraction, metric
recomputation, or model selection occurs in this reporting section.

Missing samples are reported explicitly rather than replaced with placeholders.

In [ ]:
# Utilities for checkpoint grids.
CHECKPOINT_PREVIEW_INDEX = 0
MAX_CHECKPOINTS_PER_PAGE = 6
RESULTS_PLOTS_DIR = PLOTS_DIR

def _first_readable_image(directory, preview_index=0):
    directory = Path(directory)
    preferred_names = [
        f"{preview_index:04d}.png",
        f"gen_{preview_index:04d}.png",
    ]
    candidates = [directory / name for name in preferred_names]
    candidates.extend(sorted(directory.glob("*.png")))

    seen = set()
    for path in candidates:
        if path in seen:
            continue
        seen.add(path)
        if not path.exists():
            continue
        try:
            with Image.open(path) as image:
                return path, image.convert("L").copy()
        except Exception as exc:
            warnings.warn(f"Unreadable image; trying fallback: {path} ({exc})")
    return None, None

def _load_checkpoint_preview_records():
    if not Path(EVAL_METRICS_PATH).exists():
        warnings.warn("Checkpoint metrics not found; skipping the per-checkpoint image grid.")
        return []

    with Path(EVAL_METRICS_PATH).open(encoding="utf-8") as handle:
        payload = json.load(handle)

    best_id = Path(BEST_CHECKPOINT).name if "BEST_CHECKPOINT" in globals() else None
    if best_id is None and payload:
        best_id = min(payload, key=lambda row: row.get("avg_FID", math.inf)).get("ckpt_name")

    records = []
    for row in sorted(payload, key=lambda item: int(item.get("step", 0))):
        checkpoint_id = row["ckpt_name"]
        checkpoint_dir = Path(EVAL_DIR) / checkpoint_id
        records.append({
            "checkpoint_id": checkpoint_id,
            "title": checkpoint_id.replace("checkpoint-", "ckpt "),
            "order": int(row.get("step", len(records))),
            "is_best": checkpoint_id == best_id,
            "negative_dir": checkpoint_dir / "negative",
            "positive_dir": checkpoint_dir / "positive",
        })
    return records

def _draw_checkpoint_sample_grid(records, output_path, page_index):
    n_checkpoints = len(records)
    if n_checkpoints == 0:
        return None

    fig_width = max(6.0, 2.2 * n_checkpoints)
    fig, axes = plt.subplots(
        2,
        n_checkpoints,
        figsize=(fig_width, 5.0),
        squeeze=False,
        constrained_layout=True,
    )

    fallbacks = []
    for col, record in enumerate(records):
        for row_index, (class_label, directory_key) in enumerate([
            ("Negative", "negative_dir"),
            ("Positive", "positive_dir"),
        ]):
            axis = axes[row_index, col]
            image_path, image = _first_readable_image(
                record[directory_key],
                preview_index=CHECKPOINT_PREVIEW_INDEX,
            )
            if image is None:
                axis.text(0.5, 0.5, "missing\nimage", ha="center", va="center", fontsize=8)
                axis.set_facecolor("#f2f2f2")
            else:
                axis.imshow(np.asarray(image), cmap="gray", vmin=0, vmax=255)
                expected = (
                    Path(record[directory_key]) / f"{CHECKPOINT_PREVIEW_INDEX:04d}.png",
                    Path(record[directory_key]) / f"gen_{CHECKPOINT_PREVIEW_INDEX:04d}.png",
                )
                if image_path not in expected:
                    fallbacks.append((record["checkpoint_id"], class_label, image_path.name))

            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(bool(record["is_best"]))
                spine.set_edgecolor("crimson")
                spine.set_linewidth(2.0)

            if col == 0:
                axis.set_ylabel(class_label, fontsize=11, fontweight="bold")

        title = record["title"]
        if record["is_best"]:
            title = f"{title}\nBEST"
        axes[0, col].set_title(
            title,
            fontsize=9,
            color="crimson" if record["is_best"] else "black",
            fontweight="bold" if record["is_best"] else "normal",
        )

    fig.suptitle(
        f"Images generated per checkpoint - index {CHECKPOINT_PREVIEW_INDEX} - page {page_index}",
        fontsize=13,
    )
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", output_path)

    if fallbacks:
        print("Image fallbacks used:")
        for checkpoint_id, class_label, filename in fallbacks[:12]:
            print(f"  {checkpoint_id} / {class_label}: {filename}")
        if len(fallbacks) > 12:
            print(f"  ... altri {len(fallbacks) - 12} fallback")
    return output_path

In [ ]:
# Generate checkpoint pages only.
def plot_checkpoint_sample_pages():
    records = _load_checkpoint_preview_records()
    if not records:
        return []

    saved_paths = []
    for page, start in enumerate(range(0, len(records), MAX_CHECKPOINTS_PER_PAGE), start=1):
        chunk = records[start:start + MAX_CHECKPOINTS_PER_PAGE]
        page_path = RESULTS_PLOTS_DIR / f"checkpoint_samples_negative_positive_page_{page:02d}.png"
        saved_path = _draw_checkpoint_sample_grid(chunk, page_path, page_index=page)
        if saved_path is not None:
            saved_paths.append(saved_path)
    return saved_paths

plot_checkpoint_sample_pages()